# Dataset Descriptions — `Final/data/raw/` (M5 Forecasting, Walmart)

| File | Rows | Cols | Size | Grain |
|---|---|---|---|---|
| `calendar.csv` | 1,969 | 14 | 0.1 MB | one row per day |
| `sales_train_validation.csv` | 30,490 | 1,919 | 120 MB | item x store, wide (d_1–d_1913) |
| `sales_train_evaluation.csv` | 30,490 | 1,947 | 122 MB | item x store, wide (d_1–d_1941) |
| `sell_prices.csv` | 6,841,121 | 4 | 203 MB | store x item x week |
| `sample_submission.csv` | 60,980 | 29 | 5 MB | id x 28 forecast days |
| `m5_sales_last_365_days.csv` | 11,128,850 | 8 | 879 MB | item x store x day, long (d_1577–d_1941) |

**Join keys:** sales `d_N` <-> `calendar.d`; `calendar.wm_yr_wk` <-> `sell_prices.wm_yr_wk`; `store_id` + `item_id` link sales and prices.

### 1. `calendar.csv` (~100 KB)
Date lookup with events and SNAP (food-stamp) flags.
- **Rows:** 1,969 (one per day, 2011-01-29 – 2016-06-19) | **Columns:** 14

| Column | Description |
|---|---|
| `date` | Calendar date (YYYY-MM-DD) |
| `wm_yr_wk` | Walmart year-week id (join to `sell_prices`) |
| `weekday`, `wday` | Day name and number (1 = Saturday) |
| `month`, `year` | Month and year |
| `d` | Day id `d_1`…`d_1969` (matches sales column names) |
| `event_name_1/2`, `event_type_1/2` | Holiday/event name and type (Sporting, Cultural, National, Religious); mostly null |
| `snap_CA/TX/WI` | 1 if SNAP purchases allowed that day in the state, else 0 |

### 2. `sales_train_validation.csv` (~120 MB)
Wide-format daily unit sales per item per store.
- **Rows:** 30,490 (3,049 items x 10 stores) | **Columns:** 6 + 1,913 day columns = 1,919
- **Days covered:** `d_1` – `d_1913` (2011-01-29 – 2016-04-24)

| Column | Description |
|---|---|
| `id` | Unique series id, `<item_id>_<store_id>_validation` |
| `item_id` | Item id, e.g. `HOBBIES_1_001` |
| `dept_id` | Department (7: HOBBIES_1/2, HOUSEHOLD_1/2, FOODS_1/2/3) |
| `cat_id` | Category (3: HOBBIES, HOUSEHOLD, FOODS) |
| `store_id` | Store (10: CA_1–4, TX_1–3, WI_1–3) |
| `state_id` | State (CA, TX, WI) |
| `d_1` … `d_1913` | Units sold on that day (integers, many zeros -> intermittent demand) |

Hierarchy: total -> state -> store -> category -> department -> item. Used for training; validation target is `d_1914–d_1941`.

### 3. `sales_train_evaluation.csv` (~122 MB)
Same layout as `sales_train_validation.csv`, extended by 28 days (the validation-period actuals are revealed).
- **Rows:** 30,490 | **Columns:** 6 + 1,941 day columns = 1,947
- **Days covered:** `d_1` – `d_1941` (2011-01-29 – 2016-05-22)
- Ids end in `_evaluation`. Used for the final-stage forecast of `d_1942–d_1969`.

### 4. `sell_prices.csv` (~203 MB)
Weekly selling price of each item in each store.
- **Rows:** 6,841,121 | **Columns:** 4 (only weeks where the item was sold in that store appear)

| Column | Description |
|---|---|
| `store_id` | Store id (join to sales files) |
| `item_id` | Item id (join to sales files) |
| `wm_yr_wk` | Walmart year-week id (join to `calendar.wm_yr_wk`) |
| `sell_price` | Price in USD for that week |

Join key to sales: `store_id` + `item_id` + (`d` -> `calendar.wm_yr_wk`).

### 5. `sample_submission.csv` (~5 MB)
Template for Kaggle submission: forecasts for the next 28 days per series.
- **Rows:** 60,980 (30,490 `_validation` ids + 30,490 `_evaluation` ids) | **Columns:** 29

| Column | Description |
|---|---|
| `id` | Series id ending in `_validation` or `_evaluation` |
| `F1` … `F28` | Forecast units for each of the 28 horizon days (placeholder zeros here) |

`_validation` rows forecast `d_1914–d_1941`; `_evaluation` rows forecast `d_1942–d_1969`.

### 6. `m5_sales_last_365_days.csv` (~879 MB)
Long-format (melted) version of the last 365 days of sales. **Not part of the original M5 release.**
- **Rows:** 11,128,850 (= 30,490 series x 365 days) | **Columns:** 8
- **Days covered:** `d_1577` – `d_1941`

| Column | Description |
|---|---|
| `id` | Series id, e.g. `HOBBIES_1_001_CA_1_evaluation` |
| `item_id`, `dept_id`, `cat_id`, `store_id`, `state_id` | Hierarchy keys (same as sales_train files) |
| `day` | Day id `d_N` (join to `calendar.d`) |
| `sales` | Units sold that day |

Note: file is large — read with `usecols`/`dtype`/`chunksize` or use `category` dtypes.